# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [67]:
team_setup = {
    'team_name': 'CAM Team',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Clairine', 'github': 'https://github.com/clairinelou'},
        {'name': 'Malia', 'github': 'https://github.com/maliasolomon'},
        {'name': 'Anika', 'github': 'https://github.com/anikashah2006'},
    ],
    'repo_url': 'https://github.com/clairinelou/walmart-hurricane-project',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/clairinelou/walmart-hurricane-project/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Clairine',           # nice to have
    'meeting_time': 'Tuesday 6:30-7:30 PM',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['Unit price is an object', '30 duplicate rows', '886 missing time stamps', 'timestamps stored as text', 'some SKUs have more than 1 name'],
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [68]:
!git clone https://github.com/clairinelou/walmart-hurricane-project.git

fatal: destination path 'walmart-hurricane-project' already exists and is not an empty directory.


In [69]:
import subprocess

REPO_PATH = 'walmart-hurricane-project'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

273d6b7 Charter.md
0123a35 Create CHARTER.md


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [70]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Walmart Transaction Data Cleaning  

In [71]:
import pandas as pd

transactions = pd.read_csv(
    "/content/drive/MyDrive/walmart_transactions.csv"
)

In [72]:
def remove_duplicates(df):
    return df.drop_duplicates().copy()


def clean_text_and_ids(df):
    text_columns = [
        "transaction_id", "store_id", "timestamp",
        "sku", "product_name", "category", "unit_price"
    ]

    for col in text_columns:
        df[col] = df[col].astype(str).str.strip()

    # Convert blank strings to missing values
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    # Standardize store IDs
    df["store_id"] = (
        df["store_id"]
        .str.upper()
        .str.replace("_", "-", regex=False)
        .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
    )

    df["sku"] = df["sku"].str.upper()

    return df


def consolidate_skus(df):
    # Comprehensive canonical SKU mapping covering all products
    global_sku_map = {
        # Pop-Tarts
        'SKU#459812': 'PT-12',
        'POPTART-STRAW': 'PT-12',
        'PT_STRAWBERRY': 'PT-12',
        'PT-12': 'PT-12',

        # 24pk Water
        'WATER-24': 'WATER-24PK',
        'WATER-24PK': 'WATER-24PK',
        'H2O-24': 'WATER-24PK',
        'WATER_24PK': 'WATER-24PK',

        # Emergency Candles
        'CANDLE-EMRG': 'CANDLE-EMER',
        'CNDL-EM': 'CANDLE-EMER',
        'CANDLE-EMER': 'CANDLE-EMER',

        # Heavy Duty Tarp
        'HDTARP-01': 'TARP-HD',
        'TARP_HEAVY': 'TARP-HD',
        'TARP-HD': 'TARP-HD',

        # Playing Cards
        'CARDS-01': 'PLAYING-CARDS',
        'PLAYING-CARDS': 'PLAYING-CARDS',
        'CARDS-PLAYING': 'PLAYING-CARDS',

        # Roku Stick
        'ROKU-STD': 'ROKU-STICK',
        'ROKU_STICK': 'ROKU-STICK',
        'ROKU-STICK': 'ROKU-STICK',

        # Portable Charger
        'CHRG-PORT': 'CHARGER-PORT',
        'PORT-CHARGER': 'CHARGER-PORT',
        'CHARGER-PORT': 'CHARGER-PORT',

        # Bread
        'BREAD-WH': 'BREAD-WHITE',
        'BREAD_WHITE': 'BREAD-WHITE',
        'BREAD-WHITE': 'BREAD-WHITE',

        # Peanut Butter
        'PNUTBTR-16': 'PEANUT-BUTTER-16',
        'PB-16': 'PEANUT-BUTTER-16',
        'PEANUT-BUTTER-16': 'PEANUT-BUTTER-16',

        # Chips
        'CHIPS-VP': 'CHIPS-VARIETY',
        'CHIPVARIETY': 'CHIPS-VARIETY',
        'CHIPS-VARIETY': 'CHIPS-VARIETY',

        # First Aid
        'FAID-01': 'FIRST-AID-01',
        'FIRSTAID-KIT': 'FIRST-AID-01',
        'FIRST-AID-01': 'FIRST-AID-01',

        # Paper Towels
        'PAPERTOWEL6': 'PAPER-TOWELS-6',
        'PT_6PK': 'PAPER-TOWELS-6',
        'PAPER-TOWELS-6': 'PAPER-TOWELS-6',

        # Toilet Paper
        'TOILETPAPER12': 'TOILET-PAPER-12',
        'TP-12': 'TOILET-PAPER-12',
        'TOILET-PAPER-12': 'TOILET-PAPER-12',

        # Diapers
        'DIAP-S3': 'DIAPERS-SIZE3',
        'DIAPERS-3': 'DIAPERS-SIZE3',
        'DIAPERS-SIZE3': 'DIAPERS-SIZE3',

        # Baby Formula
        'BABYFORMULA': 'BABY-FORMULA-01',
        'BFORM-01': 'BABY-FORMULA-01',
        'BABY-FORMULA-01': 'BABY-FORMULA-01',

        # Batteries
        'BAT-AA4': 'BATT-AA-4PK',
        'BATT-100': 'BATT-AA-4PK',
        'BATT_AA_4PK': 'BATT-AA-4PK',
        'BATT-AA-4PK': 'BATT-AA-4PK',

        # Soup
        'CNSOUP-01': 'SOUP-CAN-01',
        'SOUP-CAN-01': 'SOUP-CAN-01',

        # Beer
        'LTBEER24': 'BEER-24PK',
        'BEER-24': 'BEER-24PK',
        'BEER-24PK': 'BEER-24PK'
    }

    # Assign canonical SKUs, keeping original SKU if not defined in map
    df["canonical_sku"] = df["sku"].map(global_sku_map).fillna(df["sku"])
    return df


def clean_timestamps(df):
    df["timestamp_original"] = df["timestamp"]

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed",
        dayfirst=False,
        errors="coerce"
    )

    return df


def clean_prices(df):
    df["unit_price"] = pd.to_numeric(
        df["unit_price"]
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    return df


def clean_categories(df):
    category_mapping = {
        "emergency supplies": "emergency",
        "groceries": "grocery",
        "games & entertainment": "entertainment",
        "household essentials": "household",
        "baby products": "baby",
        "breakfast foods": "breakfast",
        "alcoholic beverages": "alcohol",
        "health & wellness": "health"
    }

    df["category"] = (
        df["category"].str.lower().replace(category_mapping)
    )

    return df


def clean_quantities(df):
    df["quantity_invalid"] = df["quantity"] < 0

    df["quantity"] = (
        df["quantity"]
        .mask(df["quantity_invalid"], pd.NA)
        .astype("Int64")
    )

    return df

We need to further evaluate the SKU mapping because we think there should only be 20 but we have 31.

In [73]:
def clean_transactions(df):
    cleaned = df.copy()

    cleaned = remove_duplicates(cleaned)
    cleaned = clean_text_and_ids(cleaned)
    cleaned = consolidate_skus(cleaned)
    cleaned = clean_timestamps(cleaned)
    cleaned = clean_prices(cleaned)
    cleaned = clean_categories(cleaned)
    cleaned = clean_quantities(cleaned)

    return cleaned


transactions_clean = clean_transactions(transactions)

In [74]:
transactions_clean = clean_transactions(transactions)

print("Rows before:", len(transactions))
print("Rows after:", len(transactions_clean))
print("Rows removed:", len(transactions) - len(transactions_clean))

Rows before: 25562
Rows after: 25532
Rows removed: 30


In [75]:
transactions_clean

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_sku,timestamp_original,quantity_invalid
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,grocery,1,3.15,PEANUT-BUTTER-16,2024-09-24T09:48:13,False
1,TXN000759,FL-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,grocery,1,2.02,SOUP-CAN-01,2024-09-13 20:30:20,False
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,household,2,10.47,TOILET-PAPER-12,2024-09-17 10:06:00,False
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,electronics,1,28.59,ROKU-STICK,2024-09-11 06:29:08,False
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,alcohol,3,18.82,BEER-24PK,2024-09-26 15:59:42,False
...,...,...,...,...,...,...,...,...,...,...,...
25557,TXN020201,FL-105,2024-09-03 18:28:07,DIAPERS-3,Diapers Size 3,baby,2,24.32,DIAPERS-SIZE3,2024-09-03T18:28:07,False
25558,TXN014835,FL-239,2024-09-26 08:47:08,CARDS-01,Playing Cards,entertainment,3,5.00,PLAYING-CARDS,09-26-2024 08:47:08,False
25559,TXN002434,FL-239,2024-09-16 16:39:00,PAPERTOWEL6,Paper Towels 6-Roll,household,2,7.78,PAPER-TOWELS-6,16-Sep-2024 16:39,False
25560,TXN014739,FL-710,2024-09-25 17:08:35,CHRG-PORT,PORTABLE CHARGER,electronics,1,24.18,CHARGER-PORT,2024-09-25 17:08:35,False


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [76]:
step1 = remove_duplicates(transactions.copy())
step2 = clean_text_and_ids(step1.copy())
step2_sku = consolidate_skus(step2.copy())
step3 = clean_timestamps(step2_sku.copy())
step4 = clean_prices(step3.copy())
step5 = clean_categories(step4.copy())
step6 = clean_quantities(step5.copy())

text_changed = (
    step1.fillna("").astype(str)
    != step2.fillna("").astype(str)
).any(axis=1).sum()

decision_log = pd.DataFrame([
    {
        "Decision": "Remove exact duplicates",
        "Reason": "Avoid counting the same transaction twice.",
        "Rows affected": len(transactions) - len(step1),
        "Rows removed": len(transactions) - len(step1)
    },
    {
        "Decision": "Clean text and IDs",
        "Reason": "Remove extra spaces, handle blanks, and standardize IDs.",
        "Rows affected": text_changed,
        "Rows removed": 0
    },
    {
        "Decision": "Consolidate SKUs",
        "Reason": "Map messy or redundant raw SKUs to canonical product codes.",
        "Rows affected": (step2_sku["sku"] != step2_sku["canonical_sku"]).sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Convert timestamps to datetime",
        "Reason": "Make different date formats usable for time-based analysis.",
        "Rows affected": step3["timestamp"].notna().sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Keep missing timestamps",
        "Reason": "The correct dates are unknown. exclude these from dated analyses.",
        "Rows affected": step3["timestamp"].isna().sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Convert prices to numeric",
        "Reason": "Remove dollar signs and commas so prices can be used in calculations.",
        "Rows affected": step4["unit_price"].notna().sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Standardize category labels",
        "Reason": "Combine equivalent labels so categories are grouped consistently.",
        "Rows affected": (
            step4["category"] != step5["category"
        ]).sum(),
        "Rows removed": 0
    },
    {
        "Decision": "Replace negative quantities with missing values",
        "Reason": "The correct quantities cannot be verified; exclude them from volume calculations.",
        "Rows affected": step6["quantity_invalid"].sum(),
        "Rows removed": 0
    }
])

display(decision_log)

,Decision,Reason,Rows affected,Rows removed
0,Remove exact duplicates,Avoid counting the same transaction twice.,30,30
1,Clean text and IDs,"Remove extra spaces, handle blanks, and standa...",6402,0
2,Consolidate SKUs,Map messy or redundant raw SKUs to canonical p...,16374,0
3,Convert timestamps to datetime,Make different date formats usable for time-ba...,24648,0
4,Keep missing timestamps,The correct dates are unknown. exclude these f...,884,0
5,Convert prices to numeric,Remove dollar signs and commas so prices can b...,25532,0
6,Standardize category labels,Combine equivalent labels so categories are gr...,19112,0
7,Replace negative quantities with missing values,The correct quantities cannot be verified; exc...,130,0


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [77]:
# Calculate and print the SKU consolidation statistics
starting_distinct_skus = transactions_clean["sku"].nunique()
canonical_distinct_products = transactions_clean["canonical_sku"].nunique()

# Identify groups sizes
group_sizes = (
    transactions_clean[["sku", "canonical_sku"]]
    .drop_duplicates()
    .groupby("canonical_sku")
    .size()
)

largest_group_sku = group_sizes.idxmax()
largest_group_size = group_sizes.max()

print(f"Starting distinct SKUs: {starting_distinct_skus}")
print(f"Distinct canonical products after consolidation: {canonical_distinct_products}")
print(f"Largest consolidated group: {largest_group_sku} (with {largest_group_size} different raw SKU formats consolidated)\n")

print("=== Showcase of Consolidated Pop-Tarts Family ===")
showcase_skus = ['SKU#459812', 'POPTART-STRAW', 'PT_STRAWBERRY', 'PT-12']
display(
    transactions_clean.loc[
        transactions_clean["sku"].isin(showcase_skus),
        ["sku", "canonical_sku", "product_name"]
    ].drop_duplicates()
)

Starting distinct SKUs: 45
Distinct canonical products after consolidation: 31
Largest consolidated group: PT-12 (with 4 different raw SKU formats consolidated)

=== Showcase of Consolidated Pop-Tarts Family ===


,sku,canonical_sku,product_name
21,PT_STRAWBERRY,PT-12,pop-tarts strawberry
27,SKU#459812,PT-12,POPTART-STRAWBERRY
29,PT_STRAWBERRY,PT-12,Strawberry Pop-Tart
47,PT-12,PT-12,Pop Tarts - Strawberry
52,POPTART-STRAW,PT-12,Strawberry Pop-Tart
123,POPTART-STRAW,PT-12,pop-tarts strawberry
138,PT-12,PT-12,POPTART-STRAWBERRY
142,SKU#459812,PT-12,Pop-Tarts Strawberry
148,PT_STRAWBERRY,PT-12,POPTART-STRAWBERRY
165,POPTART-STRAW,PT-12,POPTART-STRAWBERRY


This is just a starting point of the SKU work we are doing. There seems to be some underlying issues we need to work to debug.

This is just starter code, and we are going to go back and make edits as we progress through our project.

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [78]:
import pandas as pd, numpy as np, requests, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)           # backoff: 2s, 4s
            continue
        if r.status_code in RETRY_STATUSES:    # 429 or 5xx: wait and retry
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                   # any other bad status fails loudly
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])   # string -> datetime on the API side
        frame['source'] = 'open-meteo'
        return frame

# Saved copy of the same data (Orlando, Sept 1-30, 2024), used only if the API is down
FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=30),
    'precipitation_sum': [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    'windspeed_10m_max': [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                          15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                          19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    'source': 'fallback sample',
})

# The call
try:
    weather = daily_weather('2024-09-01', '2024-09-30')
except requests.RequestException as err:
    print('API call failed:', err)
    weather = FALLBACK

print('source:', weather['source'].iloc[0])
print(weather.dtypes)
weather.head()

sales_daily = (
    transactions_clean
    .dropna(subset=['timestamp'])
    .assign(date=lambda d: d['timestamp'].dt.normalize())
    .groupby('date', as_index=False)
    .agg(orders=('transaction_id', 'nunique'))
)

merged = sales_daily.merge(weather, on='date', how='left', indicator=True)

assert len(merged) == len(sales_daily), 'the join changed the row count'
assert merged['orders'].sum() == sales_daily['orders'].sum(), 'the orders total changed'
print(merged['_merge'].value_counts())
print('sales days with no weather:', merged.loc[merged['_merge'] == 'left_only', 'date'].dt.date.tolist())
merged.head()

source: open-meteo
date                 datetime64[ns]
precipitation_sum           float64
windspeed_10m_max           float64
source                       object
dtype: object
_merge
both          30
left_only      0
right_only     0
Name: count, dtype: int64
sales days with no weather: []


,date,orders,precipitation_sum,windspeed_10m_max,source,_merge
0,2024-09-01,552,3.0,18.8,open-meteo,both
1,2024-09-02,587,12.4,13.5,open-meteo,both
2,2024-09-03,582,24.8,22.8,open-meteo,both
3,2024-09-04,572,22.1,15.5,open-meteo,both
4,2024-09-05,573,22.7,15.2,open-meteo,both


Unsure if we should use only Orlando longitude and latitudeor all the stores' locations longitude and latitude.

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [79]:
# TODO

Not Started

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [80]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': len(transactions_clean),        # an integer
    'decisions_logged': len(decision_log),                 # 8 decisions logged
    'canonical_products': transactions_clean["canonical_sku"].nunique(), # distinct products after consolidation
    'weather_api_rows': len(merged),                       # rows returned by your API call
    'notebook_runs_clean': True,                           # Notebook is running clean!
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25532
  done - decisions_logged = 8
  done - canonical_products = 31
  done - weather_api_rows = 30
  done - notebook_runs_clean = True

required items complete: True
progress items done: 5 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** One thing that is likely to go wrong and impact our project is confusing amongst the three of us. Because we are working on one notebook, at times independently, it's very possible that we will not properly communicate to each other what we are all doing. As a result, there could be overwritten cells, duplicated analyses, or someone changing variable names that breaks code someone else writes later in the notebook. This could result in our notebook not running cleanly, and confusion about where something is broken as a result of us not knowing what the others are doing.


**Plan, with an owner and a date:** Moving forward, instead of us editing the same notebook, each of us will work in our own copy of the notebook and merge finished pieces into the main notebook. By Tuesday, Malia will sit up the main notebook that is only edited when pieces are entirely complete. By Wednesday, Anika will write a shared task list that says who is working on which piece. Then, on Sundays Malia and Anika will send their finished pieces to Clairine, who will add them to the main notebook and run it top to bottom to make sure nothing breaks

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | In progress| Anika |
| Q2 SKU consolidation | In progress | Clairine |
| Q3 weather correlation | In progress | Malia |
| Q4 store and geography | Not started | Malia |
| Q5 signal or artifact | Not started | Clairine |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.